# 🚀 CinemaBox 1-Click AI Cloud Server (5 TB Google Storage)
### Powered by Qwen 2.5 32B Instruct (128k Context | Zero Compilation | Verified Ready)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/princesharmaofficial1-hub/cinemabox-colab/blob/main/deploy_cinemabox_ai_v2.ipynb)

---

### 🧹 STEP 0: Wipe Colab Junk Disk
Click the **Play** button below to ensure maximum free local disk space.

In [ ]:
#@title 🧹 Clean Colab Root Disk { vertical-output: true }
print("🧹 Cleaning Colab root disk...")
!rm -rf /root/.cache/* /tmp/* /content/*.deb /content/sample_data /root/.local/share/Trash/*
print("\n✅ Local disk cleaned!")
!df -h /


--- 
### ⚡ STEP 1: 1-Click Launch AI Cloud Server
Reuses your permanently downloaded **18.49 GB model in 5 TB Google Drive** (takes **0 seconds** to download), links CUDA 12 GPU runtimes, verifies port 8000 readiness, and provides your live Cloudflare URL.

In [ ]:
#@title 🚀 1-Click Launch AI Server & Cloudflare Tunnel { vertical-output: true }
import os
import sys
import glob
import time
import subprocess
import re
import urllib.request
import site

print("="*70)
print("🚀 STARTING CINEMABOX CLOUD AI SERVER...")
print("="*70)

# 0. Create 10 GB Swap File on Colab's 200 GB disk to prevent OOM
print("\n[0/4] 🛡️ Configuring 10 GB Virtual Memory (Zero-OOM Protection)...")
!test -f /swapfile || (fallocate -l 10G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile)
print("      ✅ 10 GB Virtual Memory Swap active!")

# 1. Mount 5 TB Google Drive
print("\n[1/4] 📁 Mounting your 5 TB Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/CinemaBox_Models'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"      ✅ Connected to 5 TB Google Drive: {DRIVE_DIR}")

# 2. Setup CUDA 12 Runtime & Dependencies
print("\n[2/4] ⚡ Installing CUDA 12 Runtime & GPU Engine...")
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12

cuda_lib_dirs = set()
for p in site.getsitepackages():
    nvidia_dir = os.path.join(p, 'nvidia')
    if os.path.exists(nvidia_dir):
        for root, dirs, files in os.walk(nvidia_dir):
            if any(f.endswith('.so') or '.so.' in f for f in files):
                cuda_lib_dirs.add(root)

for d in glob.glob('/usr/local/cuda*/lib64'):
    cuda_lib_dirs.add(d)

with open('/etc/ld.so.conf.d/99-nvidia-cuda12.conf', 'w') as f:
    for d in cuda_lib_dirs:
        f.write(d + '\n')

for d in cuda_lib_dirs:
    subprocess.run(f"cp -sf {d}/*.so* /usr/lib/x86_64-linux-gnu/ 2>/dev/null || true", shell=True)
    subprocess.run(f"cp -sf {d}/*.so* /usr/lib/ 2>/dev/null || true", shell=True)

subprocess.run(["ldconfig"])

# Verify libcudart.so.12
import ctypes
try:
    ctypes.CDLL("libcudart.so.12")
    print("      ✅ CUDA 12 runtime successfully linked!")
except Exception as e:
    print(f"      ⚠️ Dynamic linker warning: {e}")

!pip install -q "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!pip install -q pydantic-settings sse-starlette starlette-context fastapi uvicorn requests jinja2

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

try:
    import llama_cpp
    print("      ✅ GPU Engine, CUDA Drivers & Cloudflare ready!")
except Exception as e:
    print(f"      ❌ llama_cpp import check: {e}")

# 3. Model in 5 TB Drive
print("\n[3/4] 📦 Checking 5 TB Google Drive Model Storage...")
MODEL_FILENAME = "Qwen2.5-32B-Instruct-Q4_K_M.gguf"
MODEL_URL = "https://huggingface.co/bartowski/Qwen2.5-32B-Instruct-GGUF/resolve/main/Qwen2.5-32B-Instruct-Q4_K_M.gguf"
model_path = os.path.join(DRIVE_DIR, MODEL_FILENAME)

if os.path.exists(model_path) and os.path.getsize(model_path) > 1024*1024*100:
    print(f"      🎯 Reusing permanently cached model in 5 TB Drive: {MODEL_FILENAME}")
    print(f"         Size: {os.path.getsize(model_path)/(1024**3):.2f} GB")
else:
    print(f"      📥 Streaming {MODEL_FILENAME} directly to your 5 TB Google Drive...")
    !wget -c --show-progress -O "{model_path}" "{MODEL_URL}"
    print(f"      ✅ Saved permanently to: {model_path}")

# 4. Launch OpenAI Server & Verify Port 8000
print("\n[4/4] 🌐 Starting Verified OpenAI-Compatible Server & Cloudflare Tunnel...")
API_KEY = "sk-cinemabox70b"

!pkill -f llama_cpp.server > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

server_env = os.environ.copy()
extra_ld = ':'.join(cuda_lib_dirs)
server_env['LD_LIBRARY_PATH'] = extra_ld + ':' + server_env.get('LD_LIBRARY_PATH', '')

server_log = open('/content/llama_server.log', 'w')
server_cmd = [
    sys.executable, "-m", "llama_cpp.server",
    "--model", model_path,
    "--n_gpu_layers", "35",
    "--n_ctx", "4096",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--api_key", API_KEY
]

server_proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT, env=server_env)

print("      ⏳ Loading weights into GPU memory & initializing engine (takes ~90-120s)... ")
server_ready = False
# Allow up to 120 attempts * 2s = 240 seconds (4 minutes)
for attempt in range(1, 121):
    time.sleep(2)
    if server_proc.poll() is not None:
        print("\n❌ Server process exited unexpectedly! Error log:")
        with open('/content/llama_server.log', 'r') as f:
            print(f.read())
        break
    try:
        req = urllib.request.Request("http://127.0.0.1:8000/v1/models")
        req.add_header("Authorization", f"Bearer {API_KEY}")
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                print(f"      ✅ GPU Server is 100% READY and responding on port 8000! (loaded in {attempt*2}s)")
                break
    except Exception:
        if attempt % 5 == 0:
            print(f"      ... warming up CUDA layers & KV cache ({attempt*2}s elapsed)")

if not server_ready:
    print("\n❌ Server failed to respond on port 8000 after 240s. Current log:")
    with open('/content/llama_server.log', 'r') as f:
        print(f.read()[-1500:])
else:
    # Start Cloudflare tunnel ONLY after port 8000 is verified 200 OK
    print("\n      🌐 Establishing Cloudflare Global Tunnel...")
    cf_log = open('/content/cloudflared.log', 'w')
    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=cf_log,
        stderr=subprocess.STDOUT
    )

    public_url = None
    for _ in range(30):
        time.sleep(1)
        if os.path.exists('/content/cloudflared.log'):
            with open('/content/cloudflared.log', 'r') as f:
                content = f.read()
                match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
                if match:
                    public_url = match.group(0)
                    break

    if public_url:
        print("\n" + "="*70)
        print("🎉 YOUR AI SERVER IS LIVE GLOBALLY AND READY TO USE!")
        print("="*70)
        print(f"\n👉 📱 CinemaBox App Endpoint:     {public_url}/v1/chat/completions")
        print(f"👉 💻 VS Code (Roo/Continue) Base: {public_url}/v1")
        print(f"👉 🔑 API Key:                     {API_KEY}")
        print(f"👉 🧠 Active Model:                Qwen2.5-32B-Instruct")
        print("\n" + "-"*70)
        print("📱 HOW TO USE IN CINEMABOX APP:")
        print(f"   1. Open CinemaBox on your phone > Settings > On-Device Intelligence")
        print(f"   2. Tap the '141B MoE / VISION' tab")
        print(f"   3. Paste '{public_url}/v1/chat/completions' into Host API Endpoint")
        print(f"   4. Tap Save & Run Test -> You are connected!")
        print("\n💻 HOW TO USE IN VS CODE (Roo Code / Continue / Cline):")
        print(f"   Provider: OpenAI Compatible")
        print(f"   Base URL: {public_url}/v1")
        print(f"   API Key:  {API_KEY}")
        print(f"   Model:    Qwen2.5-32B-Instruct")
        print("="*70 + "\n")
        print("🟢 SERVER RUNNING ACTIVELY (Keep this Colab tab open while using the app)...")
        try:
            while True:
                if server_proc.poll() is not None:
                    print("\n❌ Server stopped! Log:")
                    with open('/content/llama_server.log', 'r') as f:
                        print(f.read()[-1500:])
                    break
                time.sleep(5)
        except KeyboardInterrupt:
            print("\n🛑 Shutting down server...")
            server_proc.terminate()
            cf_proc.terminate()
    else:
        print("❌ Cloudflare tunnel could not be established. Check /content/cloudflared.log")
